# Why Jersey Numbers Are Hard to Read in Hockey: An End-to-End Player Tracking and Identification Pipeline for PWHL Broadcast Video

## Pipeline setup
Loads the core detection model (YOLOv8), pose model (for torso cropping), and the 
referee classifier trained separately (see referee-training notebook). Functions defined 
here (`run_tracking_pipeline`, `crop_torso_pose`, `is_referee`) are the validated, final 
versions of each component — the exploration and failed attempts that led to them live in 
earlier notebooks.

In [1]:
!pip install supervision scenedetect trackers

In [31]:
from ultralytics import YOLO
import supervision as sv
from scenedetect import detect, ContentDetector
from trackers import BoTSORTTracker
import matplotlib.pyplot as plt
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from collections import defaultdict, Counter
import random
import os
import json

model = YOLO("yolov8m.pt")
pose_model_x = YOLO("yolov8x-pose.pt")

device = torch.device('cpu')

class RefereeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(num_features, 2)
    def forward(self, x):
        return self.backbone(x)

model_ref_final = RefereeClassifier().to(device)
model_ref_final.load_state_dict(torch.load("referee_classifier_final.pth", map_location=device))
model_ref_final.eval()
print("Referee classifier correctly loaded")

Referee classifier correctly loaded


In [32]:
roster = pd.read_csv("roster_pwhl_2024.csv")
print(roster.shape)
roster.head()

(58, 4)


,Name,Number,Team,Position
0,BROOKE STACEY,3,Montreal,F
1,LAURA STACEY,7,Montreal,F
2,GABRIELLE DAVID,8,Montreal,F
3,LEAH LUM,12,Montreal,F
4,JILLIAN DEMPSEY,14,Montreal,F


In [33]:
def run_tracking_pipeline(video_path, start_frame=0, lost_track_buffer=90):
    scene_list = detect(video_path, ContentDetector())
    cut_frames = set(scene[0].get_frames() for scene in scene_list)

    frame_generator = sv.get_video_frames_generator(video_path)
    tracker = BoTSORTTracker(lost_track_buffer=lost_track_buffer)
    tracklets_out = {}
    segment_idx = -1

    for frame_idx, frame in enumerate(frame_generator):
        if frame_idx < start_frame:
            continue
        if frame_idx in cut_frames:
            tracker.reset()
            segment_idx += 1

        results = model(frame, verbose=False)[0]
        detections = sv.Detections.from_ultralytics(results)
        detections = detections[detections.class_id == 0]
        tracked_detections = tracker.update(detections, frame=frame)

        for i in range(len(tracked_detections)):
            tid = int(tracked_detections.tracker_id[i])
            key = (segment_idx, tid)
            bbox = tracked_detections.xyxy[i].tolist()
            conf = float(tracked_detections.confidence[i])
            if key not in tracklets_out:
                tracklets_out[key] = []
            tracklets_out[key].append((frame_idx, bbox, conf))

    tracklets_out = {k: v for k, v in tracklets_out.items() if k[1] != -1}
    return tracklets_out, cut_frames


def upscale(crop, scale=4):
    return cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)

def crop_torso(crop, bbox, top_frac=0.25, bottom_frac=0.75):
    x1, y1, x2, y2 = bbox
    h = y2 - y1
    torso_y1 = int(h * top_frac)
    torso_y2 = int(h * bottom_frac)
    return crop[torso_y1:torso_y2, :]

def crop_torso_pose(crop, pose_model, conf_thresh=0.3, upscale_factor=4):
    big_crop = upscale(crop, scale=upscale_factor)
    result = pose_model(big_crop, verbose=False)[0]
    if len(result.boxes) == 0 or result.boxes.conf[0] < conf_thresh:
        return crop_torso(crop, [0, 0, crop.shape[1], crop.shape[0]])
    keypoints = result.keypoints.xy[0]
    shoulder_y = (keypoints[5][1] + keypoints[6][1]) / 2
    hip_y = (keypoints[11][1] + keypoints[12][1]) / 2
    if shoulder_y == 0 or hip_y == 0:
        return crop_torso(crop, [0, 0, crop.shape[1], crop.shape[0]])
    scale_back = 1 / upscale_factor
    y1 = int(shoulder_y * scale_back)
    y2 = int(hip_y * scale_back)
    if y2 <= y1:
        return crop_torso(crop, [0, 0, crop.shape[1], crop.shape[0]])
    return crop[y1:y2, :]


referee_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def is_referee(crop, threshold=0.5):
    image = Image.fromarray(crop[:, :, ::-1])
    image_t = referee_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model_ref_final(image_t)
        probs = torch.softmax(output, dim=1)
        referee_prob = probs[0][0].item()
    return referee_prob > threshold

print("All functions loaded")

All functions loaded


## Step 1: Tracking
Runs BoT-SORT + scene-cut detection on the test clip. `start_frame=254` skips this 
specific clip's broadcast intro (confirmed in earlier exploration — not a general-purpose 
constant).

In [34]:
video_path = "clip_1080p_test.mp4"

tracklets_final, cut_frames_final = run_tracking_pipeline(video_path, start_frame=254)
print(f"Detected cuts: {cut_frames_final}")
print(f"Clean tracklets: {len(tracklets_final)}")

/var/folders/y8/7cx9c1ds4rb5tr5v6jbw_5m40000gn/T/ipykernel_58466/2043011856.py:3: DeprecationWarning: get_frames() is deprecated, use the `frame_num` property instead.
  cut_frames = set(scene[0].get_frames() for scene in scene_list)


Detected cuts: {0, 57, 254}
Clean tracklets: 110


## Step 2: Filter out referees
Classify each tracklet as referee/not before spending compute on jersey-number reading — 
referees are never useful for player identification, so filtering them out here avoids 
running legibility/PARSeq on tracklets that would just fail anyway.

In [35]:
video = sv.get_video_frames_generator(video_path)
targets_ref = {}
for key, entries in tracklets_final.items():
    n = len(entries)
    mid_entry = entries[n // 2]
    targets_ref.setdefault(mid_entry[0], []).append((key, mid_entry[1]))

referee_predictions = {}
for idx, frame in enumerate(video):
    if idx in targets_ref:
        for key, bbox in targets_ref[idx]:
            x1, y1, x2, y2 = map(int, bbox)
            crop = frame[y1:y2, x1:x2]
            if crop.shape[0] < 20 or crop.shape[1] < 20:
                continue
            referee_predictions[key] = is_referee(crop)
    if idx > max(targets_ref.keys()):
        break

referee_keys = [k for k, v in referee_predictions.items() if v]
player_keys = [k for k, v in referee_predictions.items() if not v]

print(f"Classified as referees: {len(referee_keys)}")
print(f"Classified as non-referee: {len(player_keys)}")

Classified as referees: 23
Classified as non-referee: 87


## Step 3: Team ID — extracting torso color
Average LAB color of the pose-cropped torso region, computed for one mid-tracklet frame 
per tracklet. (This single-frame version turns out to be unstable — see the comparison 
below.)

In [36]:
video = sv.get_video_frames_generator(video_path)
targets_color_final = {}
for key in player_keys:
    entries = tracklets_final[key]
    n = len(entries)
    mid_entry = entries[n // 2]
    targets_color_final.setdefault(mid_entry[0], []).append((key, mid_entry[1]))

def avg_color_lab(frame, bbox):
    x1, y1, x2, y2 = map(int, bbox)
    crop = frame[y1:y2, x1:x2]
    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB)
    return lab.reshape(-1, 3).mean(axis=0)

tracklet_colors_final = {}
for idx, frame in enumerate(video):
    if idx in targets_color_final:
        for key, bbox in targets_color_final[idx]:
            x1, y1, x2, y2 = map(int, bbox)
            crop = frame[y1:y2, x1:x2]
            torso = crop_torso_pose(crop, pose_model_x)
            if torso.shape[0] < 5 or torso.shape[1] < 5:
                continue
            tracklet_colors_final[key] = avg_color_lab(torso, [0, 0, torso.shape[1], torso.shape[0]])
    if idx > max(targets_color_final.keys()):
        break

print(f"Colors extracted for {len(tracklet_colors_final)} tracklets")

Colors extracted for 87 tracklets


In [37]:
from sklearn.cluster import KMeans

keys_list_final = list(tracklet_colors_final.keys())
X_final = np.array([tracklet_colors_final[k] for k in keys_list_final])

kmeans_final = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels_final = kmeans_final.fit_predict(X_final)

for cluster_id in range(2):
    count = sum(cluster_labels_final == cluster_id)
    print(f"Cluster {cluster_id}: {count} tracklets")

Cluster 0: 35 tracklets
Cluster 1: 52 tracklets


## Loading the labeled test set
Ground-truth labels (jersey number, confirmed team) from manual annotation done in an 
earlier notebook, exported to JSON so they don't need to be re-labeled here.

In [38]:
with open("test_set_labels.json") as f:
    export_data = json.load(f)

labels = {tuple(map(int, k.split("_"))): v for k, v in export_data['labels'].items()}
team_labels = {tuple(map(int, k.split("_"))): v for k, v in export_data['team_labels'].items()}
numeric_keys = [k for k, v in labels.items() if v.isdigit()]

print(f"Loaded: {len(labels)} labels, {len(team_labels)} confirmed teams, {len(numeric_keys)} with number")

Loaded: 112 labels, 19 confirmed teams, 19 with number


## Checking Team ID against ground truth (single-frame color)
Note: tracklet segment numbering differs between this notebook and the one where labels 
were created (0 here vs. 2 originally), due to how `start_frame` interacts with scene-cut 
counting — `translate_key` reconciles this. 

Result: single-frame color clustering only agreed with the confirmed team on 12/19 
(63%) — not reliable enough on its own.

In [39]:
def translate_key(old_key, old_segment=2, new_segment=0):
    seg, tid = old_key
    if seg == old_segment:
        return (new_segment, tid)
    return old_key

for key in numeric_keys:
    translated = translate_key(key)
    team = team_labels.get(key, team_labels.get(translated, "UNKNOWN"))
    if translated in keys_list_final:
        idx_in_list = keys_list_final.index(translated)
        cluster = cluster_labels_final[idx_in_list]
        print(f"{key} -> {translated}: number {labels[key]}, real team {team}, cluster {cluster}")
    else:
        print(f"{key} -> {translated}: NOT FOUND in todays tracklets")

(2, 1) -> (0, 1): number 27, real team Minnesota, cluster 0
(2, 15) -> (0, 15): number 21, real team Montreal, cluster 1
(2, 0) -> (0, 0): number 29, real team Montreal, cluster 1
(2, 3) -> (0, 3): number 13, real team Minnesota, cluster 1
(2, 4) -> (0, 4): number 21, real team Montreal, cluster 1
(2, 5) -> (0, 5): number 13, real team Minnesota, cluster 0
(2, 13) -> (0, 13): number 27, real team Minnesota, cluster 0
(2, 25) -> (0, 25): number 13, real team Minnesota, cluster 1
(2, 23) -> (0, 23): number 21, real team Montreal, cluster 1
(2, 47) -> (0, 47): number 9, real team Minnesota, cluster 1
(2, 45) -> (0, 45): number 29, real team Montreal, cluster 1
(2, 55) -> (0, 55): number 27, real team Minnesota, cluster 1
(2, 62) -> (0, 62): number 2, real team Montreal, cluster 1
(2, 71) -> (0, 71): number 21, real team Montreal, cluster 1
(2, 69) -> (0, 69): number 27, real team Minnesota, cluster 1
(2, 63) -> (0, 63): number 29, real team Montreal, cluster 1
(2, 78) -> (0, 78): number 2

## Step 3b: Averaging color over multiple frames instead of one
A single frame's torso crop can be thrown off by an imperfect pose-crop (e.g. capturing 
too much ice background) — averaging LAB color over 5 frames per tracklet reduces that 
noise.

In [40]:
targets_multi = {}
for key in player_keys:
    entries = tracklets_final[key]
    n = len(entries)
    step = max(1, n // 5)
    sample_entries = entries[::step][:5]
    for (frame_idx, bbox, conf) in sample_entries:
        targets_multi.setdefault(frame_idx, []).append((key, bbox))

video = sv.get_video_frames_generator(video_path)
colors_per_key = defaultdict(list)
for idx, frame in enumerate(video):
    if idx in targets_multi:
        for key, bbox in targets_multi[idx]:
            x1, y1, x2, y2 = map(int, bbox)
            crop = frame[y1:y2, x1:x2]
            torso = crop_torso_pose(crop, pose_model_x)
            if torso.shape[0] < 5 or torso.shape[1] < 5:
                continue
            colors_per_key[key].append(avg_color_lab(torso, [0, 0, torso.shape[1], torso.shape[0]]))
    if idx > max(targets_multi.keys()):
        break

tracklet_colors_avg = {k: np.mean(v, axis=0) for k, v in colors_per_key.items() if len(v) > 0}
print(f"Colors average for {len(tracklet_colors_avg)} tracklets")

Colors average for 87 tracklets


Re-running the same 2-cluster K-means on the averaged colors.

In [41]:
keys_list_avg = list(tracklet_colors_avg.keys())
X_avg = np.array([tracklet_colors_avg[k] for k in keys_list_avg])

kmeans_avg = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels_avg = kmeans_avg.fit_predict(X_avg)

for cluster_id in range(2):
    count = sum(cluster_labels_avg == cluster_id)
    print(f"Cluster {cluster_id}: {count} tracklets")

Cluster 0: 41 tracklets
Cluster 1: 46 tracklets


In [42]:
for key in numeric_keys:
    translated = translate_key(key)
    team = team_labels.get(key)
    if translated in keys_list_avg:
        idx_in_list = keys_list_avg.index(translated)
        cluster = cluster_labels_avg[idx_in_list]
        print(f"{translated}: number {labels[key]}, real team {team}, cluster {cluster}")
    else:
        print(f"{translated}: NOT FOUND")

(0, 1): number 27, real team Minnesota, cluster 1
(0, 15): number 21, real team Montreal, cluster 0
(0, 0): number 29, real team Montreal, cluster 1
(0, 3): number 13, real team Minnesota, cluster 1
(0, 4): number 21, real team Montreal, cluster 0
(0, 5): number 13, real team Minnesota, cluster 1
(0, 13): number 27, real team Minnesota, cluster 1
(0, 25): number 13, real team Minnesota, cluster 1
(0, 23): number 21, real team Montreal, cluster 0
(0, 47): number 9, real team Minnesota, cluster 1
(0, 45): number 29, real team Montreal, cluster 0
(0, 55): number 27, real team Minnesota, cluster 0
(0, 62): number 2, real team Montreal, cluster 0
(0, 71): number 21, real team Montreal, cluster 0
(0, 69): number 27, real team Minnesota, cluster 1
(0, 63): number 29, real team Montreal, cluster 0
(0, 78): number 2, real team Montreal, cluster 0
(0, 94): number 25, real team Montreal, cluster 0
(0, 89): number 22, real team Minnesota, cluster 1


## Step 4: Jersey number reading (legibility → torso crop → PARSeq → majority vote)
Runs the final, validated reading pipeline on the 19 ground-truth tracklets only (not all 
87 player tracklets) — this is enough to get a real accuracy figure without re-running the 
full clip's worth of compute. Full-body crops go to the legibility classifier (it expects 
uncropped players); only frames it accepts as legible are then torso-cropped and passed to 
PARSeq.

In [43]:
test_dir_integrated = "/tmp/test_integrated/imgs"
os.makedirs(test_dir_integrated, exist_ok=True)

translated_keys = {translate_key(k): k for k in numeric_keys}

frame_lookup_int = {}
for new_key in translated_keys:
    for (frame_idx, bbox, conf) in tracklets_final[new_key]:
        frame_lookup_int.setdefault(frame_idx, []).append((new_key, bbox))

video = sv.get_video_frames_generator(video_path)
saved_int = defaultdict(int)
for idx, frame in enumerate(video):
    if idx not in frame_lookup_int:
        continue
    for key, bbox in frame_lookup_int[idx]:
        x1, y1, x2, y2 = map(int, bbox)
        crop = frame[y1:y2, x1:x2]
        if crop.shape[0] < 20 or crop.shape[1] < 20:
            continue
        fname = f"{key[0]}_{key[1]}_{idx:04d}.png"
        cv2.imwrite(f"{test_dir_integrated}/{fname}", crop)
        saved_int[key] += 1

print("Crops saved by tracklet:")
for k in saved_int:
    print(k, "->", saved_int[k])

Crops saved by tracklet:
(0, 0) -> 55
(0, 1) -> 95
(0, 3) -> 9
(0, 4) -> 4
(0, 5) -> 2
(0, 13) -> 131
(0, 15) -> 99
(0, 23) -> 451
(0, 25) -> 105
(0, 45) -> 295
(0, 47) -> 210
(0, 55) -> 191
(0, 62) -> 66
(0, 63) -> 737
(0, 69) -> 337
(0, 71) -> 46
(0, 78) -> 424
(0, 89) -> 292
(0, 94) -> 190


In [44]:
!cd /Users/rebecamerazturriza/anaconda_projects/jersey-number-pipeline && python3 infer_legibility.py \
  --images_dir /tmp/test_integrated/imgs \
  --model_path models/legibility_resnet34_hockey_20240201.pth \
  --arch resnet34 \
  --result_file /tmp/test_integrated/legibility.json

/opt/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Escribió 3739 resultados en /tmp/test_integrated/legibility.json


In [45]:
with open("/tmp/test_integrated/legibility.json") as f:
    legibility_int = json.load(f)

legible_by_tracklet_int = defaultdict(list)
for fname, is_legible in legibility_int.items():
    if is_legible == 1:
        parts = fname.replace(".png", "").split("_")
        key = (int(parts[0]), int(parts[1]))
        legible_by_tracklet_int[key].append(fname)

print("Legible crops by tracklet:")
for key in translated_keys:
    print(key, "->", len(legible_by_tracklet_int.get(key, [])))

Legible crops by tracklet:
(0, 1) -> 78
(0, 15) -> 41
(0, 0) -> 55
(0, 3) -> 0
(0, 4) -> 1
(0, 5) -> 1
(0, 13) -> 6
(0, 25) -> 63
(0, 23) -> 114
(0, 47) -> 3
(0, 45) -> 82
(0, 55) -> 0
(0, 62) -> 35
(0, 71) -> 23
(0, 69) -> 47
(0, 63) -> 190
(0, 78) -> 125
(0, 94) -> 89
(0, 89) -> 41


In [46]:
torso_dir_int = "/tmp/test_integrated_torso/imgs"
os.makedirs(torso_dir_int, exist_ok=True)

entries_by_key_frame_int = {}
for key in translated_keys:
    entries_by_key_frame_int[key] = {e[0]: e[1] for e in tracklets_final[key]}

frame_lookup_torso_int = {}
for key, fnames in legible_by_tracklet_int.items():
    for fname in fnames:
        parts = fname.replace(".png", "").split("_")
        frame_idx = int(parts[2])
        bbox = entries_by_key_frame_int[key][frame_idx]
        frame_lookup_torso_int.setdefault(frame_idx, []).append((fname, bbox))

video = sv.get_video_frames_generator(video_path)
saved_torso_int = 0
for idx, frame in enumerate(video):
    if idx not in frame_lookup_torso_int:
        continue
    for fname, bbox in frame_lookup_torso_int[idx]:
        x1, y1, x2, y2 = map(int, bbox)
        crop = frame[y1:y2, x1:x2]
        torso = crop_torso_pose(crop, pose_model_x)
        if torso.shape[0] < 5 or torso.shape[1] < 5:
            continue
        cv2.imwrite(f"{torso_dir_int}/{fname}", torso)
        saved_torso_int += 1
    if idx > max(frame_lookup_torso_int.keys()):
        break

print(f"Crops of torso saved: {saved_torso_int} of {sum(len(v) for v in legible_by_tracklet_int.values())} legible")

Crops of torso saved: 993 of 994 legible


In [47]:
!cd /Users/rebecamerazturriza/anaconda_projects/jersey-number-pipeline && conda run -n parseq2 python3 str.py \
  "models/parseq_epoch=3-step=95-val_accuracy=98.7903-val_NED=99.3952.ckpt" \
  --data_root /tmp/test_integrated_torso \
  --inference \
  --result_file /tmp/test_integrated_torso/predictions.json \
  --device cpu

Additional keyword arguments: {'charset_test': '0123456789'}
/opt/anaconda3/envs/parseq2/lib/python3.9/site-packages/lightning_utilities/core/imports.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources

100%|██████████| 993/993 [00:19<00:00, 50.76it/s]


In [48]:
with open("/tmp/test_integrated_torso/predictions.json") as f:
    predictions_int = json.load(f)

votes_by_tracklet_int = defaultdict(list)
for fname, pred in predictions_int.items():
    parts = fname.replace(".png", "").split("_")
    key = (int(parts[0]), int(parts[1]))
    avg_conf = sum(pred['confidence']) / len(pred['confidence'])
    votes_by_tracklet_int[key].append((pred['label'], avg_conf))

correct_count = 0
for new_key, old_key in translated_keys.items():
    real = labels[old_key]
    votes = votes_by_tracklet_int.get(new_key, [])
    high_conf = [label for label, conf in votes if conf > 0.8]
    if high_conf:
        predicted = Counter(high_conf).most_common(1)[0][0]
    elif votes:
        predicted = Counter([v[0] for v in votes]).most_common(1)[0][0]
    else:
        predicted = "N/A"
    correct = (predicted == real)
    correct_count += correct
    print(f"{new_key}: real={real}, predicted={predicted}, {'✓' if correct else '✗'} ({len(votes)} votes)")

print(f"\nAccuracy if pipeline integrated: {correct_count}/{len(translated_keys)} = {correct_count/len(translated_keys):.1%}")

(0, 1): real=27, predicted=25, ✗ (78 votes)
(0, 15): real=21, predicted=21, ✓ (41 votes)
(0, 0): real=29, predicted=20, ✗ (55 votes)
(0, 3): real=13, predicted=N/A, ✗ (0 votes)
(0, 4): real=21, predicted=7, ✗ (1 votes)
(0, 5): real=13, predicted=9, ✗ (1 votes)
(0, 13): real=27, predicted=27, ✓ (6 votes)
(0, 25): real=13, predicted=3, ✗ (63 votes)
(0, 23): real=21, predicted=4, ✗ (114 votes)
(0, 47): real=9, predicted=4, ✗ (3 votes)
(0, 45): real=29, predicted=29, ✓ (82 votes)
(0, 55): real=27, predicted=N/A, ✗ (0 votes)
(0, 62): real=2, predicted=21, ✗ (35 votes)
(0, 71): real=21, predicted=4, ✗ (22 votes)
(0, 69): real=27, predicted=27, ✓ (47 votes)
(0, 63): real=29, predicted=20, ✗ (190 votes)
(0, 78): real=2, predicted=2, ✓ (125 votes)
(0, 94): real=25, predicted=15, ✗ (89 votes)
(0, 89): real=22, predicted=22, ✓ (41 votes)

Accuracy if pipeline integrated: 6/19 = 31.6%


# Conclusion

**Objective:** This project extends an earlier one. The first project built a detection and jersey-OCR pipeline on single PWHL frames — YOLO reliably found players, but a general-purpose OCR model performed poorly (9.7% exact accuracy, no detection at all in over 80% of cases on 288 hand-labeled crops), and a follow-up sample found non-frontal player orientation in 79 of 80 number-visible crops, the best-supported explanation found for why OCR struggled even on decent-quality crops. That project's own conclusion named the next steps: a properly-sized labeled dataset, a second OCR engine for comparison, or a purpose-trained model instead of a general-purpose one — this project pursues the last of those three, while also extending single-frame detection into full player tracking. As before, this targets PWHL footage specifically — a league where, as far as I could find, no published work has tackled this problem, unlike the NHL. It draws on two reference works: Vats, Walters, Fani, Clausi & Zelek's tracking-and-identification methodology (the source of the tracklet-based approach used here) and Koshkina's jersey-number-recognition pipeline (legibility classifier + PARSeq), used as a reference implementation to test against PWHL data.

**Methodology:** Built a full pipeline on a 1080p PWHL clip: BoT-SORT tracking (with camera motion compensation) + PySceneDetect scene-cut detection → a custom-trained referee classifier (ResNet18, transfer learning, leave-one-game-out validated) to filter non-players → K-means team clustering on pose-cropped torso color → Koshkina's legibility classifier + hockey-fine-tuned PARSeq for jersey-number reading, aggregated per tracklet via confidence-weighted majority vote. Evaluated against a manually labeled 19-tracklet test set (8 known players, cross-checked against official team rosters), plus a comparison on NHL footage from the McGill Hockey Player Tracking Dataset.

**Key findings along the way:**
- **Source resolution mattered, but wasn't the whole story.** The first clip was 360p (average player box height 50px); the same broadcast is available at 1080p (137px). Moving to 1080p raised the share of frames the legibility classifier accepted from 15% to 26% and made numbers readable to the eye, but a known ground-truth player was still misread with high confidence — resolution alone did not fix reading accuracy.
- **BoT-SORT (with camera motion compensation) reduced tracklet fragmentation** from 153 to 102 tracklets on the same clip compared to ByteTrack. The fragmentation wasn't occlusion, it was the tracker misreading normal camera panning and hard scene cuts as identity loss. A hand-built scene-cut detector was also tried and abandoned: it couldn't distinguish real cuts from fast camera movement.
- **Two pipeline stages expect different crops of the same player, and conflating them caused a silent failure.** The legibility classifier expects full-body crops (26% of frames legible vs. 0.3% with a pre-cropped torso), while PARSeq, a Scene Text Recognition model that resizes input to a fixed wide/short aspect ratio, needs a pose-cropped torso; feeding it full-body crops gave 0% accuracy (0/19) before this was corrected.
- **Uncalibrated PARSeq confidence was repeatedly unreliable** as a filter, with real cases of 0.9+ confidence on incorrect readings.
- **Three hand-crafted heuristics failed to separate referees from players** (color saturation, grayscale standard deviation, stripe-crossing counts) because source-footage blur smooths out the sharp light/dark transitions each one depends on. A small CNN trained via transfer learning on ~1,700 examples from 5 different games succeeded where the heuristics didn't, generalizing to unseen games at 91.3–98.0% (leave-one-game-out).
- **Team-color clustering was unreliable from a single tracklet frame** (63% agreement with confirmed team labels): one imperfect torso crop, e.g. capturing too much ice background, could throw off the whole estimate. Averaging color across 5 frames per tracklet substantially improved agreement to 89.5% (17/19).
- **A direct comparison against NHL footage** (the McGill dataset, Koshkina's training-data source) gave 50% jersey-reading accuracy (4/8) vs. 31.6% on PWHL (6/19), suggestive of a possible domain gap, though both samples are small.

**Results:**

| Stage | Metric | Result |
|---|---|---|
| Tracking | Tracklets on same 360p clip, ByteTrack → BoT-SORT | 153 → 102 |
| Tracking | Clean tracklets on final 1080p clip | 110 |
| Referee filtering | Accuracy on unseen games (leave-one-game-out) | 98.0% / 91.3% |
| Team ID | Agreement with confirmed team, single frame → averaged | 63% → 89.5% (17/19) |
| Jersey reading (PWHL) | Tracklet-level accuracy | 31.6% (6/19) |
| Jersey reading (NHL) | Tracklet-level accuracy | 50.0% (4/8) |

Note: Koshkina's published 91.4% is evaluated only on images her legibility classifier already accepted as legible ("considering only legible images"), and is image-level rather than tracklet-level. Our tracklet-level figures count tracklets with zero legible frames as automatic failures, so the raw numbers above are not directly comparable to hers without matching that methodology.

Relative to the 9.7% exact accuracy observed in the first project, the 31.6% tracklet-level accuracy obtained here provides contextual evidence that incorporating temporal and pipeline-level information can recover some useful identity information, although the two metrics were obtained under different evaluation protocols (single-frame exact match vs. tracklet-level majority vote) and should not be interpreted as a direct improvement measurement.

**Answer to the core question — why are jersey numbers hard to read in hockey?** Not for one reason, but for several stacked ones:
1. **Viewing angle and posture:** broadcast cameras rarely see the number head-on, and players are constantly turned, crouched, or tangled with others. This wasn't re-quantified in this project, but the qualitative pattern matches — and is consistent with — the first project's measured finding (79/80 non-frontal crops in a hand selected sample).
2. **Resolution and motion blur:** at 360p the number is only a few pixels tall; even at 1080p, blur smooths the edges that both hand-crafted heuristics and OCR models rely on.
3. **Physical occlusion:** in the source footage itself, a folded jersey can hide part of a digit ("29" read as "20"), and some digit shapes are ambiguous even to a human at this resolution ("43" read as "13"). The McGill dataset's own documentation reports that about 27.5% of its tracklets cannot be identified by jersey number at all, even with manual annotation, and that about 13.5% of player instances occlude another player.
4. **Pipeline-level pitfalls:** mismatched crop conventions between stages and overconfident model scores can silently destroy accuracy without any obvious error.
5. **Possible domain gap:** the reading model performed better on the NHL footage it was trained on (50% vs. 31.6%), consistent with a possible domain gap — though with sample sizes this small and no independent control for confounds like broadcast style or compression, this is suggestive rather than confirmed (see Limitations).

The referee-filtering stage generalized across games (leave-one-game-out, unseen-game accuracy 91.3–98.0%), while tracking and team-ID produced usable intermediate outputs on the evaluated PWHL footage — neither was tested for cross-game generalization the way the referee classifier was. Jersey reading is the stage with the clearest gap, and some of its failures are ceilings of the source footage rather than fixable bugs.

**Limitations:**
- Small test sets (19 PWHL tracklets, 8 NHL tracklets): accuracy figures are directional evidence, not precise estimates.
- The domain-gap hypothesis is suggestive, not confirmed; it would need a larger, matched evaluation (legible-only, tracklet-level, multiple PWHL and NHL games).
- Team ID and the final pipeline were validated on a single clip; the referee classifier on 5 games. Performance on the full 22-game PWHL dataset is untested.
- The pose-crop fallback (fixed top/bottom fractions) still triggers when pose detection fails, and caused at least one Team ID miss; not fully resolved.
- PARSeq confidence scores were not recalibrated for this domain.
- Camera angle, broadcast production style, and resolution are confounded in the NHL/PWHL comparison and weren't tested independently.
- Roster-based plausibility filtering (using team ID plus known rosters) could correct some reading errors but was not implemented.